<img src="./static/imo_health.png" alt="IMO Health Logo" width="300"/>

---

# IMO Knowledge Graph Traversal

This notebook demonstrates querying the IMO Knowledge Graph GraphQL endpoint and formatting mappings in a table.

Endpoint: `https://api.imohealth.com/knowledgegraph/graphql/`

## Step 1: Install Packages and Load Configuration

Copy `config.json.template` to `config.json` and provide IMO client credentials.

In [ ]:
%pip install requests pandas boto3 --quiet

import json
import pathlib
import boto3
from botocore.exceptions import BotoCoreError, ClientError
import requests
import pandas as pd
from IPython.display import display

candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json'
]
cfg_path = next((p for p in candidates if p.exists()), None)

if cfg_path is None:
    raise FileNotFoundError('config.json not found. Copy config.json.template to config.json and fill in credentials.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

kg_cfg = cfg.get('knowledge_graph', {})
TOKEN_URL = kg_cfg.get('token_url', 'https://api.imohealth.com/oauth/token')
GRAPHQL_URL = kg_cfg.get('graphql_url', 'https://api.imohealth.com/knowledgegraph/graphql/')

SSM_PARAMS = {
    'client_id':     '/imo/knowledge_graph/client_id',
    'client_secret': '/imo/knowledge_graph/client_secret',
}

def _get_from_ssm(param_name: str) -> str:
    ssm = boto3.client('ssm')
    response = ssm.get_parameter(Name=param_name, WithDecryption=True)
    return response['Parameter']['Value']

print(f'Loaded config from: {cfg_path.resolve()}')

credentials = {}
for key, param_name in SSM_PARAMS.items():
    try:
        credentials[key] = _get_from_ssm(param_name)
        print(f'  [{key}] loaded from SSM: {param_name}')
    except (BotoCoreError, ClientError) as e:
        fallback = kg_cfg.get(key, '')
        credentials[key] = fallback
        print(f'  [{key}] SSM failed ({e}), using config.json fallback')

CLIENT_ID     = credentials['client_id']
CLIENT_SECRET = credentials['client_secret']

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError('client_id/client_secret could not be resolved from SSM or config.json.')

print('Credentials ready.')

## Step 2: Get OAuth Access Token

In [ ]:
def get_token(client_id: str, client_secret: str) -> str:
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'audience': 'https://api.imohealth.com'
    }
    resp = requests.post(TOKEN_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()['access_token']

access_token = get_token(CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {access_token}', 'Content-Type': 'application/json'}

print('Token acquired (prefix):', access_token[:20] + '...')

In [ ]:
from IPython.display import HTML

# ── Shared display utilities ──────────────────────────────────────────────────

TABLE_STYLE = [
    {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]},
]

def df_to_html(df):
    return df.style.set_table_styles(TABLE_STYLE).to_html()

def items_to_html(label, items, columns):
    h = f'<p style="margin:8px 0 2px 0"><b>{label} ({len(items)})</b></p>'
    if items:
        df = pd.DataFrame([{c: i.get(c, '') for c in columns} for i in items])
        h += df_to_html(df)
    else:
        h += '<p style="margin:0 0 6px 12px"><i>(none)</i></p>'
    return h

def refinements_section_html(label, refinements):
    h = f'<p style="margin:6px 0 2px 0"><b>{label}</b></p>'
    if refinements:
        df = pd.DataFrame([{
            'code':        r.get('code', ''),
            'title':       r.get('title', ''),
            'group_code':  r.get('group', {}).get('code', ''),
            'group_title': r.get('group', {}).get('title', ''),
        } for r in refinements])
        h += df_to_html(df)
    else:
        h += '<p style="margin:0 0 6px 12px"><i>(none)</i></p>'
    return h

print('Display utilities loaded.')

In [ ]:
# ── Pagination helper ─────────────────────────────────────────────────────────

PAGE_SIZE = 1000

def fetch_all_pages(code, domain, field, fragment_type, sub_fields, extra_args=""):
    """
    Paginate a single list field on a Lexical node using offset/size.
    Returns (lexical_title, all_items).
    """
    all_items = []
    lexical_title = None
    offset = 0

    while True:
        args = f"offset: {offset}, size: {PAGE_SIZE}"
        if extra_args:
            args = f"{extra_args}, {args}"

        query = f'''{{
  lexical(code: "{code}", domain: {domain}) {{
    title
    ... on {fragment_type} {{
      {field}({args}) {{
        {sub_fields}
      }}
    }}
  }}
}}'''
        resp = requests.post(GRAPHQL_URL, headers=headers, json={'query': query}, timeout=60)
        resp.raise_for_status()
        result = resp.json()

        lexical_data = result.get('data', {}).get('lexical', {})
        if not lexical_data:
            break

        if lexical_title is None:
            lexical_title = lexical_data.get('title', '')

        page = lexical_data.get(field, [])
        all_items.extend(page)

        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    return lexical_title, all_items

print(f'Pagination helper loaded (PAGE_SIZE={PAGE_SIZE}).')

## Step 2.5: Normalize a Clinical Term

Calls the IMO Precision Normalize API using the same bearer token obtained in Step 2.
The returned IMO lexical code is used in all subsequent Knowledge Graph queries.

In [ ]:
import uuid
import requests as _requests

NORMALIZE_URL = kg_cfg.get('normalize_url', 'https://api.imohealth.com/precision/normalize')
CLINICAL_TERM = 'Chest pain'
DOMAIN        = 'problem'

# Normalize API uses separate credentials from KG API
normalize_client_id = kg_cfg.get('normalize_client_id', '')
normalize_client_secret = kg_cfg.get('normalize_client_secret', '')

if normalize_client_id and normalize_client_secret:
    normalize_token = get_token(normalize_client_id, normalize_client_secret)
    print(f'Using separate Normalize credentials (client_id: {normalize_client_id[:8]}...)')
else:
    normalize_token = access_token
    print('WARNING: No separate normalize credentials found, using KG token (may get 401)')

payload = {
    'organization_id':   'IMO',
    'client_request_id': str(uuid.uuid4()),
    'preferences':       {'threshold': 0.0, 'match_field_pref': 'input_term'},
    'requests': [{
        'record_id':  str(uuid.uuid4()),
        'domain':     DOMAIN,
        'input_term': CLINICAL_TERM,
    }],
}

resp = _requests.post(
    NORMALIZE_URL,
    headers={'Authorization': f'Bearer {normalize_token}', 'Content-Type': 'application/json'},
    json=payload,
    timeout=60,
)
resp.raise_for_status()
data = resp.json()

items = (data.get('requests') or [{}])[0].get('response', {}).get('items', [])
top   = items[0] if items else {}

LEXICAL_CODE  = top.get('lexical_code', '')
LEXICAL_TITLE = top.get('lexical_title') or top.get('title', '')

if not LEXICAL_CODE:
    raise ValueError(f'Normalization returned no result for "{CLINICAL_TERM}".')

print(f'Term          : {CLINICAL_TERM}')
print(f'IMO Lexical Code  : {LEXICAL_CODE}')
print(f'IMO Lexical Title : {LEXICAL_TITLE}')

## Step 3: Query the Mappings in Knowledge Graph 

First query: retrieve code system mappings for the normalized IMO lexical code.

> **Schema note:** `mappings` is defined on `ProblemLexical` (a concrete subtype), not on the base `Lexical` interface. Use an inline fragment `... on ProblemLexical` to access it.

In [ ]:
graphql_query = f'''
query get_mappings{{
  lexical(code: "{LEXICAL_CODE}") {{
    title
    ... on ProblemLexical {{
      mappings {{
        code
        codeSystem
      }}
    }}
  }}
}}
'''

response = requests.post(GRAPHQL_URL, headers=headers, json={'query': graphql_query}, timeout=60)
response.raise_for_status()
result = response.json()

lexical  = result.get('data', {}).get('lexical', {})
title    = lexical.get('title', '')
mappings = lexical.get('mappings', [])

output = f'<h4>Lexical title: {title}</h4>'
output += items_to_html('Code Mappings', mappings, ['code', 'codeSystem'])

display(HTML(output))

print('\nRaw JSON response:')
print(json.dumps(result, indent=2))

## Step 4: Navigate Concept Hierarchy

Run a hierarchy query for lexical code `85191` and display `domainBroader`/`domainNarrower` concepts in pretty tables.

> **Schema note:** `ProblemLexical` uses `domainBroader` and `domainNarrower` (domain hierarchy fields) instead of the generic `broader`/`narrower`.

In [ ]:
# Paginate domainBroader
hier_title, domain_broader = fetch_all_pages(
    code="85191", domain="problem", field="domainBroader",
    fragment_type="ProblemLexical",
    sub_fields='code\n        title\n        ... on ProblemLexical { mappings { code codeSystem } }'
)

# Paginate domainNarrower
_, domain_narrower = fetch_all_pages(
    code="85191", domain="problem", field="domainNarrower",
    fragment_type="ProblemLexical",
    sub_fields="code\n        title"
)

print(f'domainBroader: {len(domain_broader)} items, domainNarrower: {len(domain_narrower)} items')

def _df_section(label, df, empty_msg='(none)'):
    h = f'<p style="margin:8px 0 2px 0"><b>{label}</b></p>'
    h += df_to_html(df) if not df.empty else f'<p style="margin:0 0 6px 12px"><i>{empty_msg}</i></p>'
    return h

output = f'<h4>Lexical title: {hier_title}</h4>'

broader_df = pd.DataFrame([{'code': b.get('code', ''), 'title': b.get('title', '')} for b in domain_broader])
output += _df_section('Domain Broader Concepts', broader_df)

broader_mappings_rows = []
for b in domain_broader:
    for m in b.get('mappings', []):
        broader_mappings_rows.append({
            'broader_code':  b.get('code', ''),
            'broader_title': b.get('title', ''),
            'mapping_code':  m.get('code', ''),
            'code_system':   m.get('codeSystem', '')
        })
broader_mappings_df = pd.DataFrame(broader_mappings_rows)
output += _df_section('Domain Broader Mappings', broader_mappings_df)

narrower_df = pd.DataFrame([{'code': n.get('code', ''), 'title': n.get('title', '')} for n in domain_narrower])
output += _df_section('Domain Narrower Concepts', narrower_df)

display(HTML(output))

## Step 5: Drill Down Narrower Relationships

Run a second-level hierarchy query using `domainNarrower` for the normalized IMO lexical code and show parent-to-child relationships in a pretty table.

In [ ]:
# Level 1: paginate domainNarrower for root code
drill_title, level1_narrower = fetch_all_pages(
    code="85191", domain="problem", field="domainNarrower",
    fragment_type="ProblemLexical",
    sub_fields="code\n        title"
)

# Level 2: for each level-1 child, paginate its domainNarrower
for item in level1_narrower:
    _, children = fetch_all_pages(
        code=item['code'], domain="problem", field="domainNarrower",
        fragment_type="ProblemLexical",
        sub_fields="code\n        title"
    )
    item['domainNarrower'] = children

print(f'Level 1: {len(level1_narrower)} items')

level1_df = pd.DataFrame([{
    'level1_code':  item.get('code', ''),
    'level1_title': item.get('title', ''),
    'child_count':  len(item.get('domainNarrower', []))
} for item in level1_narrower])

output = f'<h4>Lexical title: {drill_title}</h4>'
output += '<p style="margin:8px 0 2px 0"><b>Level 1 Domain Narrower Concepts (with child counts)</b></p>'
output += df_to_html(level1_df) if not level1_df.empty else '<p><i>(none)</i></p>'

drill_rows = []
for parent in level1_narrower:
    parent_code  = parent.get('code', '')
    parent_title = parent.get('title', '')
    children     = parent.get('domainNarrower', [])
    if children:
        for child in children:
            drill_rows.append({'parent_code': parent_code, 'parent_title': parent_title,
                               'child_code': child.get('code', ''), 'child_title': child.get('title', '')})
    else:
        drill_rows.append({'parent_code': parent_code, 'parent_title': parent_title,
                           'child_code': '', 'child_title': ''})

drill_df = pd.DataFrame(drill_rows)
output += '<p style="margin:8px 0 2px 0"><b>Drill-Down Domain Narrower Relationships (Level 1 → Level 2)</b></p>'
output += df_to_html(drill_df) if not drill_df.empty else '<p><i>(none)</i></p>'

display(HTML(output))

## Step 6: Query Allowed Refinements and Groups

Run a refinements query for lexical code `85191` and show each concept with its IMO Health refinement group.

Use the Knowledge graph content on Concept Refinements to build Refinement workflows.

> **Schema note:** `ProblemLexical` exposes `allowedRefinements` (via `RefinementHierarchy`) and `refinementNarrower(refinements: [...])` for filtered navigation.

In [ ]:
# Paginate allowedRefinements
ref_title, allowed_refinements = fetch_all_pages(
    code="85191", domain="problem", field="allowedRefinements",
    fragment_type="ProblemLexical",
    sub_fields="code\n        title\n        group { code title }"
)

print(f'allowedRefinements: {len(allowed_refinements)} items')

refinement_rows = [{'concept_code': r.get('code', ''), 'concept_title': r.get('title', ''),
                    'group_code': r.get('group', {}).get('code', ''),
                    'group_title': r.get('group', {}).get('title', '')}
                   for r in allowed_refinements]
refinements_df = pd.DataFrame(refinement_rows)

group_summary_df = (
    refinements_df.groupby(['group_code', 'group_title'], dropna=False)
    .size().reset_index(name='concept_count')
    .sort_values(by='concept_count', ascending=False).reset_index(drop=True)
) if not refinements_df.empty else pd.DataFrame()

output = f'<h4>Lexical title: {ref_title}</h4>'

output += '<p style="margin:8px 0 2px 0"><b>Refinement Concepts with Groups</b></p>'
output += df_to_html(refinements_df) if not refinements_df.empty else '<p><i>(none)</i></p>'

output += '<p style="margin:8px 0 2px 0"><b>Refinement Group Summary</b></p>'
output += df_to_html(group_summary_df) if not group_summary_df.empty else '<p><i>(none)</i></p>'

display(HTML(output))

## Step 7: Sequential Refinements

Apply refinements sequentially — each `refinementNarrower` result is further narrowed by the next refinement code in the chain:

`75952` → narrow by `1403` → narrow by `1105` → narrow by `1112`

Each nested level needs `... on ProblemLexical` because `refinementNarrower` returns `[Lexical]`.

In [ ]:
sequential_refinements_query = '''
query get_sequential_refinements{
  lexical(code: "75952") {
    title
    ... on ProblemLexical {
      appliedRefinements(offset: 0, size: 1000) {
        code
        title
        group { code title }
      }
      allowedRefinements(offset: 0, size: 1000) {
        code
        title
        group { code title }
      }
      refinementNarrower(refinements: ["1403"], offset: 0, size: 1000) {
        code
        title
        ... on ProblemLexical {
          appliedRefinements(offset: 0, size: 1000) {
            code
            title
            group { code title }
          }
          allowedRefinements(offset: 0, size: 1000) {
            code
            title
            group { code title }
          }
          refinementNarrower(refinements: ["1105"], offset: 0, size: 1000) {
            code
            title
            ... on ProblemLexical {
              appliedRefinements(offset: 0, size: 1000) {
                code
                title
                group { code title }
              }
              allowedRefinements(offset: 0, size: 1000) {
                code
                title
                group { code title }
              }
              refinementNarrower(refinements: ["1112"], offset: 0, size: 1000) {
                code
                title
              }
            }
          }
        }
      }
    }
  }
}
'''

seq_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': sequential_refinements_query},
    timeout=60
)
seq_response.raise_for_status()
seq_result = seq_response.json()

seq_lexical = seq_result.get('data', {}).get('lexical', {})
seq_title   = seq_lexical.get('title', '')
level1      = seq_lexical.get('refinementNarrower', [])

def _level_section_html(title, node):
    """HTML block for one refinement level: divider + applied + allowed tables."""
    h = f'<hr style="margin:12px 0 4px 0"><p><b>{title}</b></p>'
    h += refinements_section_html('Applied Refinements', node.get('appliedRefinements', []))
    h += refinements_section_html('Allowed Refinements', node.get('allowedRefinements', []))
    return h

output = f'<h4>Lexical title: {seq_title}</h4>'
output += _level_section_html('Root (75952)', seq_lexical)

for l1 in level1:
    output += _level_section_html(
        f'After refinement 1403: {l1.get("code")} — {l1.get("title")}', l1
    )
    for l2 in l1.get('refinementNarrower', []):
        output += _level_section_html(
            f'After refinement 1105: {l2.get("code")} — {l2.get("title")}', l2
        )
        final = l2.get('refinementNarrower', [])
        output += '<p style="margin:6px 0 2px 0"><b>Final results after refinement 1112</b></p>'
        if final:
            df_final = pd.DataFrame([{'code': c.get('code', ''), 'title': c.get('title', '')} for c in final])
            output += df_to_html(df_final)
        else:
            output += '<p style="margin:0 0 6px 12px"><i>(none)</i></p>'

display(HTML(output))

print('\nRaw JSON response:')
print(json.dumps(seq_result, indent=2))

## Step 8: Cross-Domain Relationships

Query cross-domain links between domains using the `MedicationLexical` and `ProcedureLexical` types.

**`MedicationLexical` cross-domain fields:**
- `treatedProblems` — problems for which this medication is listed as a treatment
- `causedProblems` — problems for which this medication is listed as a causative agent
- `contraindicatedProblems` — problems for which this medication is contraindicated
- `preventedProblems` — problems for which this medication is listed as a preventive agent

**`ProcedureLexical` cross-domain fields:**
- `associatedProblems` — problems that list this procedure as an associated procedure
- `interpretedFindings` — problems that interpret this procedure
- `causedProblems` — problems that list this procedure as a due-to cause

<cell_type>markdown</cell_type>### Step 8a: Medication → Problem Cross-Domain Links

Query `treatedProblems` and `causedProblems` for a `MedicationLexical` concept using pagination.

> **Note:** `preventedProblems` is deprecated (always returns []) and `contraindicatedProblems` does not exist in the current schema.

In [ ]:
MED_CODE = "114959"

# Paginate treatedProblems
med_title, treated_problems = fetch_all_pages(
    code=MED_CODE, domain="medication", field="treatedProblems",
    fragment_type="MedicationLexical",
    sub_fields="code\n        title"
)

# Paginate causedProblems
_, caused_problems = fetch_all_pages(
    code=MED_CODE, domain="medication", field="causedProblems",
    fragment_type="MedicationLexical",
    sub_fields="code\n        title"
)

print(f'treatedProblems: {len(treated_problems)} items, causedProblems: {len(caused_problems)} items')

output = f'<h4>Medication lexical title: {med_title}</h4>'
output += items_to_html('Treated Problems', treated_problems, ['code', 'title'])
output += items_to_html('Caused Problems', caused_problems, ['code', 'title'])

display(HTML(output))

### Step 8b: Procedure → Problem Cross-Domain Links

Query `associatedProblems`, `interpretedFindings`, and `causedProblems` for a `ProcedureLexical` concept.

In [ ]:
PROC_CODE = "1231"

# Paginate associatedProblems
proc_title, associated_problems = fetch_all_pages(
    code=PROC_CODE, domain="procedure", field="associatedProblems",
    fragment_type="ProcedureLexical",
    sub_fields="code\n        title"
)

# Paginate interpretedFindings
_, interpreted_findings = fetch_all_pages(
    code=PROC_CODE, domain="procedure", field="interpretedFindings",
    fragment_type="ProcedureLexical",
    sub_fields="code\n        title"
)

# Paginate causedProblems
_, proc_caused_problems = fetch_all_pages(
    code=PROC_CODE, domain="procedure", field="causedProblems",
    fragment_type="ProcedureLexical",
    sub_fields="code\n        title"
)

print(f'associatedProblems: {len(associated_problems)}, interpretedFindings: {len(interpreted_findings)}, causedProblems: {len(proc_caused_problems)}')

output = f'<h4>Procedure lexical title: {proc_title}</h4>'
output += items_to_html('Associated Problems', associated_problems, ['code', 'title'])
output += items_to_html('Interpreted Findings', interpreted_findings, ['code', 'title'])
output += items_to_html('Caused Problems', proc_caused_problems, ['code', 'title'])

display(HTML(output))